---
title: "Cost-per-click surge attribution"
author: "Andratx Bellmunt"
abstract: >
  Adapted from a real business scenario. A +18% cost-per-click surge is observed in aggregated data for 1,900+ digital advertising campaigns. Rate-mix decomposition is used to properly attribute the contribution of each individual campaign to the global figure.
format:
  html:
    code-fold: true
    self-contained: true
    html-math-method: katex
    include-after-body: _tracker.html
jupyter: python3
number-sections: false
---

# Initialization

## Imports and settings

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML

In [ ]:
# Configuration
pio.renderers.default = "plotly_mimetype+notebook_connected"

## Auxiliary functions

In [ ]:
def display_long_table(df: pd.DataFrame, format: str) -> None:
    assert format in ["jupyter", "html"], "Valid formats are 'jupyter' and 'html'"

    if format == "jupyter":
        tbl_style = "style='display:inline-block; max-height:300px; overflow-y:scroll;'"
        display(df.set_table_attributes(tbl_style))
    
    if format == "html":
        tbl_style = '<div style="overflow-y: auto; max-height: 300px;">{0}</div>'
        display(HTML(tbl_style.format(df.to_html())))

In [ ]:
def add_derived_columns(df: pd.DataFrame, mix_rate: bool=False) -> pd.DataFrame:
    df["cpc_before"] = df["cost_before"] / df["clicks_before"]
    df["cpc_after"] = df["cost_after"] / df["clicks_after"]
    df["cpc_delta"] = df["cpc_after"] - df["cpc_before"]
    df["cpc_perc_diff"] = (df["cpc_after"] / df["cpc_before"] - 1) * 100

    return df

def aggregate_data(df: pd.DataFrame, numeric_only: bool=False) -> pd.DataFrame:
    df_agg = pd.DataFrame(df.sum(numeric_only=numeric_only)).T
    df_agg["clicks_before"] = df_agg["clicks_before"].astype(int)
    df_agg["clicks_after"] = df_agg["clicks_after"].astype(int)

    return df_agg

def compute_rate_mix_effects(df: pd.DataFrame) -> pd.DataFrame:
    # Click share columns
    df["click_share_before"] = df["clicks_before"] / df["clicks_before"].sum()
    df["click_share_after"] = df["clicks_after"] / df["clicks_after"].sum()
    df["click_share_delta"] = df["click_share_after"] - df["click_share_before"]

    # Rate-mix effects columns
    df["rate_effect"] = df["click_share_after"] * df["cpc_delta"]
    df["mix_effect"] = df["click_share_delta"] * df["cpc_before"]
    df["total_effect"] = df["rate_effect"] + df["mix_effect"]

    # Percentual rate-mix effects columns
    total_cpc_before = df["cost_before"].sum() / df["clicks_before"].sum()
    df["rate_perc_effect"] = df["rate_effect"] / total_cpc_before * 100
    df["mix_perc_effect"] = df["mix_effect"] / total_cpc_before * 100
    df["total_perc_effect"] = df["rate_perc_effect"] + df["mix_perc_effect"]

    return df

# The data

We read the data set ([download as csv](https://andratx_bellmunt.github.io/portfolio/src/assets/cpc_data.csv)):

  - It contains clicks and costs data for 1,908 digital advertising campaigns

  - Data corresponds to two comparable periods, labelled "before" and "after"

In [ ]:
df_base = pd.read_csv("../assets/cpc_data.csv")
display_long_table(df_base.style.hide().format(precision=2), format="html")

\
\
We now compute the cost per click (CPC) for before and after and add the delta and the percentual difference between the two:

In [ ]:
df = add_derived_columns(df_base)
display_long_table(df.style.hide().format(precision=2), format="html")

\
\
Finally, we aggregate the data to see the global effects:

In [ ]:
df_agg = add_derived_columns(aggregate_data(df, numeric_only=True))
display(df_agg.style.hide().format(precision=2))

# What the stakeholders see and do (and why it does not work)

When looking at the aggregate data the stakeholders see that **the global CPC has increased +18%** and they are alarmed.

Due to operational constraints they cannot realistically take action on more than 100 campaigns, so they need to prioritize.

They proceed as follows:

  - Take the 100 largest campaigns in terms of clicks

  - Sort them according to percentual CPC increase
  
  - Prioritize actions according to that ranking

This strategy has two main flaws:

  - **Sorting by percentage CPC increase ignores the base CPC level.** A 20% increase on a $1 click is a +$0.20 change. A 5% increase on a $50 click is $2.50, far more impactful. Percentage change without reference to the baseline is not a measure of impact.

  - **Individual CPC increases do not necessarily imply an aggregate CPC increase.** A set of campaigns can all increase their individual CPCs while the aggregate goes down, and the reverse is equally possible. This happens when the mix of clicks across campaigns changes significantly between the two periods.

## The mix problem: Simpson's paradox

Let us illustrate with a toy example the second of the flaws that we mentioned above:

In [ ]:
df_toy_base = pd.DataFrame(
    columns=["id", "cost_before", "clicks_before", "cost_after", "clicks_after"],
    data=[
        ["A", 1000.00, 100,  200.00, 25],
        ["B",  900.00,  40,  800.00, 40]
    ]
)

df_toy = add_derived_columns(df_toy_base)

df_toy_agg = aggregate_data(df_toy)
df_toy_agg = add_derived_columns(df_toy_agg)

print("Toy example:")
display(df_toy.style.hide().format(precision=2))
print("Aggregated data:")
display(df_toy_agg.style.hide().format(precision=2))

\
Even in a simple example with only two campaigns we can see how the phenomenon of [Simpson's paradox](https://en.wikipedia.org/wiki/Simpson%27s_paradox) arises:

- Both campaigns **individual CPCs go down**: -20.00% for product A and -11.11% for product B

- However the **aggregated CPC goes up**: +13.36%

The main culprit is that **the mix between the two campaigns has completely changed**. We shall review this in more detail later.

Note that in our real case scenario –where we have 1,900+ campaigns instead of just two– these interactions become much more complex.

# First proposed technical solution (and why it does not work)

In order to navigate the problems we exposed while keeping the stakeholders language (namely "percentual CPC changes") we can borrow a tool from game theory: [Shapley values](https://en.wikipedia.org/wiki/Shapley_value).

- Shapley values measure the contribution of each individual player to a common goal

- To us, each campaign is a player and the common goal is the aggregated CPC percentual change. We want to measure how much each individual campaign contributes to it.

- One property of Shapley values is that individual contributions always add up to the final global result (*efficiency axiom*). E.g. in our toy example above, the sum of the Shapley value of A and the Shapley value of B must be +13.36.

- More in general, in our real data, we shall compute the 1,900 Shapley values and all of them would add up to +18.04.

On paper, this approach works well because we can say to stakeholders "from the global +18.04%, this campaign contributed exactly this much". However, as we will readily see, it fails to capture the real reason behind the CPC surge.

Let us get back to our toy example and compute by the corresponding Shapley values (for only two campaigns the formula is easy to apply directly):

In [ ]:
print("Shapley values for CPC percentual change:")
print(f"  Campaign A: {round(1/2 * (df_toy_agg['cpc_perc_diff'].iloc[0] + df_toy['cpc_perc_diff'].iloc[0] - df_toy['cpc_perc_diff'].iloc[1]), 2)}%")
print(f"  Campaign B: {round(1/2 * (df_toy_agg['cpc_perc_diff'].iloc[0] + df_toy['cpc_perc_diff'].iloc[1] - df_toy['cpc_perc_diff'].iloc[0]), 2)}%")

This points to B as the main culprit. However, we know from our construction of the example that A is the campaign whose behavior changed most dramatically (it lost 75% of its clicks). B did nothing wrong: its CPC actually decreased.

The reason Shapley misleads us here is that the marginal contribution of each campaign depends not only on its own CPC dynamics (rate effects), but on the volume composition of the coalition it enters (mix effects). A's dramatic volume loss distorts every coalition it participates in, and that distortion gets reflected in B's attributed contribution.

More fundamentally, CPC percentage change conflates two distinct mechanisms: individual CPC changes and click volume mix shifts. Shapley has no way to separate them, so the attribution it produces reflects both entangled together. Attributing percentage CPC change via Shapley is effectively attributing the wrong thing.

# The proper solution

As we have been hinting in previous sections, the contribution of how an individual campaign contributes the global CPC surge depends on the combination of two factors:

- How its CPC changes

- How its share of the total number of clicks shifts

If we denote by $w_i = \frac{\text{clicks}_i}{\text{total\_clicks}}$ the clicks share of campaign $i$, we get the following decomposition:

$$\Delta{\text{CPC}}_{\text{global}} = \sum_i w_i^{\text{after}}\cdot\Delta{\text{CPC}_i} + \sum_i\Delta w_i\cdot\text{CPC}_i^{\text{before}}$$


In the sum,

- The left term measures **rate effects**: whether the campaign's CPC has improved or worsened, holding clicks share fixed

- The right term measures **mix effects**: whether the clicks mix shifted toward lower or higher CPC campaigns

*Note:* If we want to keep the percentual change narrative, by dividing each term of the sum by $\text{CPC}^{\text{before}}$ we get a percentual equivalent of the rate-mix decomposition.

## Results tables

We now compute the rate-mix effects for our original data:

In [ ]:
df = compute_rate_mix_effects(add_derived_columns(df, mix_rate=True))
df_agg = add_derived_columns(aggregate_data(df, numeric_only=True))

### Aggregated results

Let us begin taking a look at the aggregated data to get a general sense of the results we obtained:

In [ ]:
display(
    df_agg.rename_axis("metric").T.rename(columns={0: "value"}).style.format(precision=2)
)

- Just as a double check that our computations are right: as expected, after aggregating, both click shares are 1 and their delta is 0.

- Note also that we get what it was expected in terms of total effects:
  
  - Total effect equals the $0.55 that we got for CPC delta

  - Total percentage effect equals the +18.04% that we got as CPC percentage difference

- Most importantly, this table shows the **the mix effect is much larger than the rate effect**:

  - Of the 55 cents, 49 are explained by mix effects and only 6 by rate effects

  - In percentual terms, of the total +18% increase, 16% comes from mix effects and only 2% from rate effects

These numbers show that the strategy followed by the stakeholders, that mainly focused on rate effects, might be missing relevant data. We shall confirm this in the next section in which we expect the results at individual campaign level.

### Individual campaigns

In [ ]:
def display_mix_rate_effects(df: pd.DataFrame) -> None:
    default_fmt = "{:.2f}"
    overrides = {
        "click_share_before": "{:.6f}",
        "click_share_after": "{:.6f}",
        "click_share_delta": "{:.6f}",
        "rate_effect": "{:.6f}",
        "mix_effect": "{:.6f}",
        "total_effect": "{:.6f}",
        "rate_perc_effect": "{:.6f}",
        "mix_perc_effect": "{:.6f}",
        "total_perc_effect": "{:.6f}",
    }

    fmt = {col: overrides.get(col, default_fmt) for col in df.select_dtypes("float").columns}

    display_long_table(
        df
        .sort_values(by="total_effect", ascending=False)
        .style
        .hide()
        .format(fmt),
        format="html"
    )

print("Results by campaign (sorted by total effect):")
display_mix_rate_effects(df)

A simplified version of this table is exactly the **deliverable that the stakeholders need**: campaigns ranked in order of their effect on the total CPC surge.

In particular we can use it to compare how much of the results are we capturing with the new strategy. Remember:

- Stakeholder's strategy:

   - Optimize the top 100 campaigns ranked by click volume, i.e. $w_i^\text{after}$

- Proposed strategy: 

   - Optimize the top 100 campaigns ranked by total effect, i.e. $w_i^{\text{after}}\cdot\Delta{\text{CPC}_i} + \Delta w_i\cdot\text{CPC}_i^{\text{before}}$

In [ ]:
df_aux = pd.DataFrame(
    [df.sort_values(by="clicks_after", ascending=False).head(100).sum(numeric_only=True).to_dict(),
    df.sort_values(by="total_effect", ascending=False).head(100).sum(numeric_only=True).to_dict()]
)[["rate_effect", "mix_effect", "total_effect", "rate_perc_effect", "mix_perc_effect", "total_perc_effect"]]

df_aux["new_cpc_delta"] = df_agg["cpc_delta"].iloc[0] - df_aux["total_effect"]
df_aux["new_cpc_perc_diff"] = df_agg["cpc_perc_diff"].iloc[0] - df_aux["total_perc_effect"]

print("Effects captured by Top 100 campaigns:")
display(
    df_aux
    .T
    .reset_index(names="effect")
    .rename(columns={0:"stakeholders_strategy", 1: "prposed_strategy"})
    .style
    .hide()
    .format(precision=2)
)

- Stakeholder's selected campaigns capture only $0.16 of the the total $0.55 CPC delta (+5.28% of the total +18.04% CPC surge)

- The proposed strategy actually *surpasses* the $0.55 CPC delta and accounts for a total effect of $0.66 (+21.87% CPC surge over the registered +18.04%). Note that this can happen because there are campaigns with negative total effect.

- Comparing the two strategies (assuming targeted campaigns are moved to 0 total effect):

  - At most stakeholders strategy can bring the CPC delta down from +$0.55 to +$0.39. The proposed strategy can bring it down to -$0.11, going beyond compensanting the surge.

  - In percentual terms, the initial strategy can reduce the results from +18.04% to +12.76% CPC surge. With the rate-mix strategy we can prevent the surge altogether and even improve the CPCs by -3.83%.

  - In summary, **the new strategy improves results by more than 4x**

## Visualizing the results

### Stakeholder's strategy

To begin with let us plot the CPC change between before and after. This essentially captures the rate effects. We plot one dot per campaign, with size adjusted (at log scale) by the number of clicks in the "after" period.

In [ ]:
def plot_cpc_comparison(df: pd.DataFrame) -> None:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["cpc_before"],
            y=df["cpc_after"],
            mode="markers",
            marker_size=df["clicks_after"].apply(lambda x: 2 * np.log(x)),
            marker_color="#009AD7",
            text=df["campaign_id"],
            name="CPC before/after"
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[0,80],
            y=[0,80],
            mode="lines",
            line=dict(color="red", dash="dash", width=1),
            name="Same CPC before/after"
        )
    )

    fig.update_layout(
        title="CPC comparison",
        width=600,
        height=600,
        xaxis_title="CPC before",
        yaxis_title="CPC after",
        yaxis_scaleanchor="x",
        yaxis_scaleratio=1,
        template="plotly_dark"
    )

    fig.show()

plot_cpc_comparison(df)


\
\
To fully replicate the stakeholders strategy let us limit the previous plot to the top 100 campaigns in terms of clicks. That is, we are taking only the campaigns with the largest $w_i^{\text{after}}$:

In [ ]:
plot_cpc_comparison(df.sort_values(by="clicks_after", ascending=False).head(100))

All the selected campaigns are close to the diagonal, so their CPC change is small. This helps visualazing the flaws that we mentioned above:

  - Rate effects $w_i^{\text{after}}\cdot\Delta{\text{CPC}}_i$ are only partially captured because although $w_i^{\text{after}}$ are high (big dots), the $\Delta{\text{CPC}}_i$ are low (dots close to the diagonal)
  
  - Mix effects are ignored altogether

## Rate-mix effects

In [ ]:
def plot_rate_mix_effect_comparison(df: pd.DataFrame) -> None:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["rate_effect"],
            y=df["mix_effect"],
            mode="markers",
            marker_size=4,
            marker_color="#009AD7",
            text=df["campaign_id"],
            name="Rate-mix effect"
        )
    )

    for i in range(-1,6):
        fig.add_trace(
            go.Scatter(
                x=[-0.02,0.02],
                y=[0.01 * i + 0.02, 0.01 * i - 0.02],
                mode="lines",
                line=dict(color="red", dash="dash", width=1),
                name=f"Total effect level",
                showlegend=i==-1
            )
        )

        fig.add_annotation(
            x=0.02,
            y=0.01 * i - 0.02, 
            text=f"{round(0.01 * i, 2)}",
            xshift=20,
            font=dict(color="red"),
            showarrow=False
        )

    fig.update_layout(
        title="Rate vs Mix effect on CPC change",
        width=600,
        height=600,
        xaxis_title="Rate effect",
        yaxis_title="Mix effect",
        #yaxis_scaleanchor="x",
        #yaxis_scaleratio=1,
        xaxis_range=[-0.021, 0.021],
        #showlegend=False,
        template="plotly_dark"
    )

    fig.show()

plot_rate_mix_effect_comparison(df)

- The plot shows the rate and mix effects of each campaign, one on each axis

- The red dashed lines show levels of equal total effect:

   - Most campaigns have a total effect between -$0.01 and +$0.01

   - Three campaigns stand out: two are above +$0.05 and another one right below +$0.04

   - Hovering over the corresponding dots we can see that these are the campaigns with ids `id_0141`, `id_0071`, `id_0024` (as expected, these agree with the top 3 campaigns in the delivarable table shown above)

### Rate effects factors

In [ ]:
def plot_rate_effect_factors(df: pd.DataFrame) -> None:
    max_abs = np.abs(df["rate_effect"]).max()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["cpc_delta"],
            y=df["click_share_after"],
            mode="markers",
            marker=dict(
                size=5,
                color=df["rate_effect"],
                colorscale="RdBu_r",
                cmin=-max_abs,
                cmax=max_abs,
                colorbar=dict(title="Rate effect")
            ),
            text=df.apply(lambda row: f"{row['campaign_id']} rate effect = {round(row['rate_effect'], 6)}", axis=1)
        )
    )

    fig.update_layout(
        title="Rate effect factors",
        width=600,
        height=600,
        xaxis_title="CPC delta",
        yaxis_title="Click share after",
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

plot_rate_effect_factors(df)

- The rate effect factors are $\Delta{\text{CPC}_i}$ (x-axis) and $w_i^{\text{after}}$ (y-axis)

- The rate effect is large (in absolute value) if both factors are large (in absolute value). In the plot this translates into being far from the axes.

- We colored the dots according to their rate effect. Note that campaigns `id_0141`, `id_0071`, `id_0024`, `id_0122` stand out. In the rate vs mix plot they are the rightmost dots, as expected.

- On the other side of things `id_0000`, `id_0064` stand out as the bluest campaings (leftmost in the rate vs mix plot). This indicates the maximum reduction in terms of rate effect.

### Mix effect factors

In [ ]:
def plot_mix_effect_factors(df: pd.DataFrame) -> None:
    max_abs = np.abs(df["mix_effect"]).max()
    
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["click_share_delta"],
            y=df["cpc_before"],
            mode="markers",
            marker=dict(
                size=5,
                color=df["mix_effect"],
                colorscale="RdBu_r",
                cmin=-max_abs,
                cmax=max_abs,
                colorbar=dict(title="Mix effect")
            ),
            text=df.apply(lambda row: f"{row['campaign_id']} mix effect = {round(row['mix_effect'], 6)}", axis=1)
        )
    )

    fig.update_layout(
        title="Mix effect factors",
        width=600,
        height=600,
        xaxis_title="Click share delta",
        yaxis_title="CPC before",
        showlegend=False,
        template="plotly_dark"
    )

    fig.show()

plot_mix_effect_factors(df)

- The rate effect factors are $\Delta{w_i}$ (x-axis) and $\text{CPC}^{\text{before}}$ (y-axis)

- The rate effect is large (in absolute value) if both factors are large (in absolute value). In the plot this translates into being far from the axes.

- We colored the dots according to their rate effect. Note that campaigns `id_0141`, `id_0071`, `id_0024` stand out. In the rate vs mix effects plot they are the topmost dots, as expected.

- Note that, this time `id_0000` and `id_0064` also show in reddish tone, indicating a high mix effect (in the rate vs mix effects plot they are well above the x-axis). Hence, these two campaigns are a perfect example on how rate effects are not enough to attribute contribution to the global CPC change.

# Final comments

## Business comments

### Segmentation

- In order to simplify the analysis and to showcase the technique we are treating all clicks equally regardless of the campaign that generated them. 

- In a real case scenario, each campaign would represent a different country or product and their clicks cannot be compared directly. 

- Providing an analysis segmented by country and/or product is advisable.


### The root business problem

- At the beginning of the notebook we mentioned that the periods "before" and "after" were comparable (indeed they correspond to equal timespans).

- However, one would observe that the cost in the "after" period actually doubled that of the "before" period.

- This is explained by the fundamental reason that originated the analysis presented here: the "after" period actually represents a time later in the month in which campaigns were overspending in order to exhaust preassigned budgets.

- The fundamental action that solved the CPC surge problem was rethinking how budgets were preassigned and putting limits on spending rates and required exhaustion rates. What our analysis provides is a list on were to prioritize such actions.


### CPC is not the only KPI

- CPC was a very valuable metric to our stakeholders because it represented real clients costs and their perception on the quality of our product (no client likes to see that they are paying +18% more for the very same product just because the month is coming to an end)

- From our business point of view, however, we need to link this KPI with other ones that account for revenue (not included here) like revenue per click (RPC) or return on investment (ROI). How much do we make from each click is key to make a proper optimization of the campaigns.

- Note that being ratios themselves, RPC and ROI changes can also be treated by means of rate-mix effect decompositions

## Technical comments

### Shapley values

- We disregarded Shapley values because in this particular application they do not capture the effects we need to account for

- Aside from that they have another complication: they are expensive to compute. Using the direct definition they require $O(2^n)$ operations. The alternative definition using permutations is even worse at $O(n!)$ operations. For $n$=1,900 campaigns applying this is totally infeasable.

- The usual approach is applying sample permutations (Monte Carlo method). If done in a smart way (see e.g. [https://arxiv.org/pdf/2010.12082](https://arxiv.org/pdf/2010.12082)) we can get good approximations in $O(k\cdot n)$ operations where $k$ is the number of samples. Rough heuristics using the central limit theorem show that with $k$=5,000 we can get very stable estimates.

- Anyhow, even if they were the right tool, this is overkill for a task that can be solved with rate-mix decomposition that requires only $O(n)$ operations and provides exact results.

### Robustness

- The rate-mix decomposition gives us an exact deterministic result for our 1,900 campaigns

- However, clicks and costs in any given period are subject to random fluctuations. A different time window would yield slightly different numbers. A natural question is therefore: how stable is our top-100 ranking?

- This can be assessed by bootstrap resampling:
  - Resample the 1,900 campaigns with replacement and recompute the full decomposition. Repeat this k times (e.g. k=1000)
  
  - For each campaign, record its inclusion frequency: how often it appears in the top 100 across the k rankings.
  
  - Campaigns with inclusion frequency above 90% are robustly prioritized. Campaigns near the boundary (e.g. ranks 80–120) may be interchangeable and should be treated with caution.

- This gives stakeholders not just a ranked list, but a stability guarantee: the top campaigns are the right ones to act on regardless of short-term fluctuations in the data.